# ME Sales Panel (Jupyter)

Same data as the Google Sheets panel and the Streamlit app: the BigQuery bridge
`css-operations.me_panel_dev_us.me_sales_panel_k_monthly` (rebuilt every 12h).

**How to use**: set `COUNTRY` in the next cell, then Run All.
On notebooks.cssinternal.com the default credentials already have BigQuery access;
locally run `gcloud auth application-default login` once.

Notes: recognized revenue (RRA/RRL/NRRA $) matures over ~2 months - the live month
shows deal-date figures until recognized loads, then switches automatically.


In [ ]:
# ---- settings ----
COUNTRY = "Middle East"   # one of: Middle East / Saudi Arabia / UAE / Kuwait / Bahrain / Qatar
START_MONTH = "2025-01-31"
BRIDGE_TABLE = "css-operations.me_panel_dev_us.me_sales_panel_k_monthly"
BQ_PROJECT = "css-operations"


In [ ]:
# ---- pull the bridge ----
import math
from decimal import Decimal
import pandas as pd
from google.cloud import bigquery

client = bigquery.Client(project=BQ_PROJECT)
_q = (
    "SELECT * FROM `" + BRIDGE_TABLE + "` "
    "WHERE month_end >= DATE '" + START_MONTH + "' "
    "AND month_end <= LAST_DAY(CURRENT_DATE(), MONTH) "
    "ORDER BY month_end, country"
)
_rows = []
for _r in client.query(_q).result():
    _d = dict(_r)
    for _k, _v in list(_d.items()):
        if isinstance(_v, Decimal):
            _d[_k] = float(_v)
    _rows.append(_d)
df = pd.DataFrame(_rows)
df["month_end"] = pd.to_datetime(df["month_end"])
d = df[df["country"] == COUNTRY].sort_values("month_end").reset_index(drop=True)
d["month_label"] = d["month_end"].dt.strftime("%b %Y")
print(f"{COUNTRY}: {len(d)} months, {df['country'].nunique()} countries in bridge, last month = {d['month_label'].iloc[-1] if len(d) else '-'}")


In [ ]:
# ---- the Panel (metrics x months) ----
def fmt(v, kind):
    try:
        if v is None or (isinstance(v, float) and math.isnan(v)):
            return ""
        if kind == "usd":
            return f"${float(v):,.0f}"
        if kind == "usd2":
            return f"${float(v):,.2f}"
        if kind == "pct":
            return f"{float(v) * 100:.1f}%"
        if kind == "num1":
            return f"{float(v):,.1f}"
        return f"{float(v):,.0f}"
    except Exception:
        return "" if v is None else str(v)

METRICS = [
    ("Sales", None, None),
    ("CWs (kitchens)", "cws", "num"),
    ("Approved Deals", "approved_deals", "num"),
    ("Avg CW Duration (months)", "cw_duration", "num1"),
    ("New Occupied Kitchens", "new_occupied_k", "num"),
    ("Revenue $", None, None),
    ("RRA $ (recognized, CW-date)", "rra_usd", "usd"),
    ("RRL $ (recognized)", "rrl_usd", "usd"),
    ("NRRA $ (net)", "nrra_usd", "usd"),
    ("RRX $ (accessed)", "xrra_usd", "usd"),
    ("RRLX $ (post-access lost)", "xrrl_usd", "usd"),
    ("NRRX $ (net accessed)", "nrrx_usd", "usd"),
    ("Gross Recurring Revenue $ (EoP)", "gross_rr_usd", "usd"),
    ("RR after MKO/MFO $ (EoP)", "rr_after_mko_mfo_usd", "usd2"),
    ("TCV $", "tcv_usd", "usd"),
    ("Approved TCV $", "approved_tcv_usd", "usd"),
    ("Churn & Net", None, None),
    ("Churns (excl. transfers)", "churns_excl_transfers", "num"),
    ("Net Adds", "net_adds", "num"),
    ("Occupancy & Sold Rates", None, None),
    ("Occupancy", "occupancy", "pct"),
    ("Occupied Kitchens", "occupied_kitchens", "num"),
    ("Total Kitchen Numbers (TKN)", "total_kitchens", "num"),
    ("Live - Sold Rate %", "live_sold_rate", "pct"),
    ("Live - Sold Rate w/ Approved %", "live_sold_rate_approved", "pct"),
    ("Vacant w/ Approved Opp", "live_vacant_appr_k", "num"),
    ("Sold Rate - All Facilities", "sold_rate_all", "pct"),
    ("Sold Rate w/ Approved - All", "net_sold_approved_rate", "pct"),
    ("Team", None, None),
    ("Sales Team Size (FTE)", "sales_team_size", "num1"),
    ("AEs", "aes", "num1"),
    ("SDRs", "sdrs", "num1"),
]

_months = d["month_label"].tolist()
_labels, _rows2 = [], []
for _lbl, _col, _kind in METRICS:
    if _col is None:
        _labels.append("- " + _lbl + " -")
        _rows2.append([""] * len(_months))
    elif _col in d.columns:
        _labels.append(_lbl)
        _rows2.append([fmt(_v, _kind) for _v in d[_col].tolist()])
panel = pd.DataFrame(_rows2, index=_labels, columns=_months)
try:
    from IPython.display import display
    display(panel.style.set_caption(COUNTRY + " - ME Sales Panel").set_properties(**{"text-align": "right"}))
except Exception:
    print(panel.to_string())


In [ ]:
# ---- Dashboard charts ----
import plotly.graph_objects as go

def _show(fig):
    try:
        get_ipython  # noqa: B018 - only show inline inside Jupyter
        fig.show()
    except NameError:
        pass

_x = d["month_label"].tolist()

def line_fig(series, title, is_pct=False):
    fig = go.Figure()
    for _lbl, _col, _color in series:
        if _col in d.columns:
            fig.add_trace(go.Scatter(x=_x, y=d[_col], mode="lines+markers", name=_lbl,
                                     line=dict(color=_color, width=2)))
    fig.update_layout(title=COUNTRY + " - " + title, height=380,
                      legend=dict(orientation="h", y=-0.25), plot_bgcolor="white")
    if is_pct:
        fig.update_yaxes(tickformat=".0%")
    return fig

_show(line_fig([("RRA $", "rra_usd", "#0F766E"), ("RRL $", "rrl_usd", "#DC2626"),
                ("NRRA $", "nrra_usd", "#2563EB")],
               "Recurring Revenue (recognized) - added / lost / net"))
_show(line_fig([("Gross RR $", "gross_rr_usd", "#0F766E"),
                ("RR after MKO/MFO $", "rr_after_mko_mfo_usd", "#7C3AED")],
               "Occupied-kitchen revenue at End of Period (gap = concession load)"))
_figb = go.Figure()
if "cws" in d.columns:
    _figb.add_trace(go.Bar(x=_x, y=d["cws"], name="CWs", marker_color="#0F766E"))
if "approved_deals" in d.columns:
    _figb.add_trace(go.Bar(x=_x, y=d["approved_deals"], name="Approved", marker_color="#94A3B8"))
_figb.update_layout(title=COUNTRY + " - CWs vs Approved Deals", barmode="group", height=380,
                    legend=dict(orientation="h", y=-0.25), plot_bgcolor="white")
_show(_figb)
_show(line_fig([("Occupancy", "occupancy", "#2563EB"), ("Live Sold Rate", "live_sold_rate", "#0F766E"),
                ("Live Sold Rate w/ Approved", "live_sold_rate_approved", "#F59E0B")],
               "Occupancy vs Live Sold Rates", is_pct=True))
print("charts built OK")
